## Seção 4.2 - Análise de tendências qualidade do ar no Brasil

Este notebook reproduz a tabela 21 e a figura 34 da seção 4.2 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Figura 34 - Mapa interativo das tendências interanuais de CO, NO₂, SO₂, MP₂,₅, MP₁₀ e O₃ no Brasil
Mapa interativo permite selecionar cada poluente e visualizar as tendências para cada estação e poluente.

A visualização dos pontos das estações com tendência positva estão com o simbolo de triângulo para baixo em **azul**, e os pontos em triângulo para cima **vermelho**, representando as estações com tendência positiva, as pontos com círculo **cinza** são as estações com p-valor do teste de Mann-Kendall maior ou igual a 0.05, ou seja, não significativos, e os pontos com círculo em **branco**, as estações com período insuficiente

Ao clicar na estação, é possível visualizar o ID estação, nome da estação, período de monitoramento disponível, anos inválidos, p-valor do teste Mann-Kendall da estação, declividade, mediana e o percentual de anual de mudança da estação de cada poluente escolhido no filtro. 

> **Pré-requisito:** este mapa depende dos arquivos `{poluente}_stations.geojson` gerados por `scripts/trend_analisys.ipynb`. Execute esse notebook primeiro (ele lê os dados diretamente da URL hospedada e salva os GeoJSONs em `_static/preprocessed_trends/`) antes de rodar a célula abaixo.

In [2]:
import re
import json
from pathlib import Path
from IPython.display import HTML
import os

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste TRENDS_DIR para a pasta onde os arquivos {poluente}_stations.geojson foram
gerados no seu computador por scripts/trend_analisys.ipynb antes de executar este
script. Como o código é compartilhado via Git, este caminho varia entre usuários. '''
TRENDS_DIR = Path("../_static/preprocessed_trends/")

POLUENTES = ["O3", "CO", "NO2", "MP25", "MP10", "SO2"]

def to_safe_key(pol):
    return re.sub(r'[^a-z0-9]+', '', pol.lower())

# Carrega, em Python, todos os GeoJSONs de tendência disponíveis localmente e
# embute os dados no HTML — evita depender de fetch() em tempo real, que não
# funciona quando o HTML é aberto como arquivo local (file://) fora de um servidor.
data = {}
for pol in POLUENTES:
    fpath = TRENDS_DIR / f"{to_safe_key(pol)}_stations.geojson"
    if fpath.exists():
        with open(fpath, encoding="utf-8") as f:
            data[pol] = json.load(f)

data_json = json.dumps(data, ensure_ascii=False)

html_template = r"""
<style>
#map-controls { display:flex; flex-wrap:wrap; gap:12px; align-items:center; margin-bottom:12px; font-family:Arial, sans-serif; }
.control { display:flex; align-items:center; gap:8px; height:40px; }
.control-label { font-size:14px; font-weight:500; color:#333; white-space:nowrap; }
.select-wrap { position:relative; display:inline-block; width:120px; height:32px; }
.select-wrap select { appearance:none; display:block; width:100%; height:100%; padding:6px 34px 6px 10px; font-size:14px; border-radius:6px; border:1px solid #999; background:#f9f9f9; cursor:pointer; }
.select-wrap::after{ content:""; position:absolute; pointer-events:none; top:50%; transform:translateY(-50%); right:10px; width:12px; height:12px; background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 10 6'><path fill='%23333' d='M0 0l5 6 5-6z'/></svg>"); background-repeat:no-repeat; background-size:12px 12px; }
.control-button { height:32px; padding:0 16px; line-height:32px; border-radius:6px; border:1px solid #005a9e; background:#0078d7; color:#fff; cursor:pointer; font-weight:600; margin-left:auto; text-align:center; }
.control-button:hover{ background:#005a9e; }
#status { font-size:13px; color:crimson; margin-left:8px; }
#leafletMap{ width:100%; height:640px; border:1px solid #ddd; display:none; margin-top:8px; }

/* legenda customizada (gradiente) */
.leaflet-control.custom-legend { background:white; padding:8px; border-radius:4px; box-shadow:0 1px 4px rgba(0,0,0,0.2); font-size:13px; }
.legend-gradient { height:12px; width:180px; border-radius:3px; margin:6px 0; display:block; }
.legend-labels { display:flex; justify-content:space-between; gap:8px; font-size:12px; }
.legend-symbol { display:inline-block; width:14px; height:14px; margin-right:4px; vertical-align:middle; border:1px solid #222; }
</style>

<div id="map-controls">
  <div class="control">
    <span class="control-label">Poluente:</span>
    <div class="select-wrap">
      <select id="sel-poll">
        <option>O3</option>
        <option>CO</option>
        <option>NO2</option>
        <option>MP25</option>
        <option>MP10</option>
        <option>SO2</option>
      </select>
    </div>
  </div>

  <div class="control" style="flex:1">
    <button id="btn-load" class="control-button">Gerar mapa</button>
    <span id="status"></span>
  </div>
</div>

<div id="leafletMap"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
(function(){
  const DATA = __DATA__;
  const selP = document.getElementById('sel-poll');
  const btn  = document.getElementById('btn-load');
  const status = document.getElementById('status');
  const mapDiv = document.getElementById('leafletMap');

  const tilesDefs = {
    "OpenStreetMap": { url: 'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', attribution:'© OpenStreetMap' },
    "CartoDB Positron": { url: 'https://cartodb-basemaps-a.global.ssl.fastly.net/light_all/{z}/{x}/{y}.png', attribution:'© OpenStreetMap, © CartoDB' },
    "Esri WorldImagery": { url: 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attribution:'Tiles © Esri' }
  };

  let map = null;
  let ptsLayer = null;
  let baseLayers = {};
  let layerControl = null;
  let legendControl = null;

  function ensureMap(){
    if(map) return;
    map = L.map('leafletMap').setView([-15.78, -47.9], 4);
    Object.keys(tilesDefs).forEach((name, idx) => {
      const def = tilesDefs[name];
      baseLayers[name] = L.tileLayer(def.url, { maxZoom: 19, attribution: def.attribution });
      if(idx===0) baseLayers[name].addTo(map);
    });
    layerControl = L.control.layers(baseLayers, {}, { collapsed: false }).addTo(map);
  }

  function hexToRgb(hex){ hex=hex.replace('#',''); if(hex.length===3) hex=hex.split('').map(c=>c+c).join(''); const bigint=parseInt(hex,16); return [(bigint>>16)&255,(bigint>>8)&255, bigint&255]; }
  function rgbToHex(r,g,b){ return '#' + [r,g,b].map(x=>{const h=x.toString(16); return h.length===1?'0'+h:h;}).join(''); }
  function interpHex(a,b,t){ const aa=hexToRgb(a), bb=hexToRgb(b); return rgbToHex(Math.round(aa[0]+(bb[0]-aa[0])*t), Math.round(aa[1]+(bb[1]-aa[1])*t), Math.round(aa[2]+(bb[2]-aa[2])*t)); }

  const darkBlue="#08306b", lightBlue="#deebf7", lightRed="#fee0d2", darkRed="#a50f15";

  function colorForPercent(val,minV,maxV){
    if(val==null || isNaN(val)) return "#666666";
    const v=Number(val);
    if(minV>=0 && maxV>=0) return interpHex(lightRed,darkRed,(v-minV)/(maxV-minV||1));
    if(minV<=0 && maxV<=0) return interpHex(darkBlue,lightBlue,(v-minV)/(maxV-minV||1));
    if(v<0) return interpHex(darkBlue,lightBlue,(v-minV)/(0-minV||1));
    return v>0?interpHex(lightRed,darkRed,(v-0)/(maxV-0||1)):"#ffffff";
  }

  function buildPopup(p){
    function fmt(v){ return (v==null||v==""||isNaN(v))?"n/a":Number(v).toFixed(3); }
    const s=p.station||p.estacao||"n/a", id=p.ID_OEMA||p.id_oema||"n/a";
    const n_valid=p.n_valid_years||0, n_invalid=p.invalid_years||"";
    const pval=("p_value" in p)?fmt(p.p_value):"n/a", slope=("slope" in p)?fmt(p.slope):"n/a";
    const median=("median" in p)?fmt(p.median):"n/a", percent=("percent_change" in p)?fmt(p.percent_change):"n/a";
    const periodo=(p.start_year&&p.end_year)?`${p.start_year} - ${p.end_year}`:"n/a";
    return `<b>ID Estação:</b> ${s}<br/><b>Estação:</b> ${id}<br/><b>Período:</b> ${periodo}<br/><b>Anos inválidos:</b> ${n_invalid}<br/><b>p-valor:</b> ${pval}<br/><b>Declividade:</b> ${slope}<br/><b>Mediana:</b> ${median}<br/><b>% mudança:</b> ${percent}`;
  }

  function addLegend(minV,maxV){
    if(legendControl){ try{ legendControl.remove(); } catch{} legendControl=null; }
    legendControl=L.control({position:'bottomright'});
    legendControl.onAdd=function(){
      const div=L.DomUtil.create('div','leaflet-control custom-legend');
      div.innerHTML=`<strong>Mudança anual (%)</strong>`;
      let gradHtml='';
      if(minV<0 && maxV>0) gradHtml=`<div class="legend-gradient" style="background:linear-gradient(to right, ${darkBlue} 0%, ${lightBlue} 49%, ${lightRed} 51%, ${darkRed} 100%);"></div>`;
      else if(maxV<=0) gradHtml=`<div class="legend-gradient" style="background:linear-gradient(to right, ${darkBlue}, ${lightBlue});"></div>`;
      else gradHtml=`<div class="legend-gradient" style="background:linear-gradient(to right, ${lightRed}, ${darkRed});"></div>`;
      div.innerHTML+=gradHtml;
      div.innerHTML+=`<div class="legend-labels"><span>${isNaN(minV)?"n/a":minV.toFixed(3)}</span><span>0</span><span>${isNaN(maxV)?"n/a":maxV.toFixed(3)}</span></div>`;
      div.innerHTML+=`<div style="margin-top:4px;"><span class="legend-symbol" style="background:#cccccc;border:1px solid #222;border-radius:50%;"></span> Não significativo (p ≥ 0.05)</div>`;
      div.innerHTML+=`<div style="margin-top:2px;"><span class="legend-symbol" style="background:#ffffff;border:1px solid #222;border-radius:50%;"></span> Período insuficiente</div>`;
      div.innerHTML+=`<div style="margin-top:2px;"><span class="legend-symbol" style="width:0;height:0;border-left:7px solid transparent;border-right:7px solid transparent;border-bottom:12px solid #a50f15;display:inline-block;"></span> Tendência positiva</div>`;
      div.innerHTML+=`<div style="margin-top:2px;"><span class="legend-symbol" style="width:0;height:0;border-left:7px solid transparent;border-right:7px solid transparent;border-top:12px solid #08306b;display:inline-block;"></span> Tendência negativa</div>`;
      return div;
    };
    legendControl.addTo(map);
  }

  btn.addEventListener('click', function(){
    try{
      status.textContent=''; mapDiv.style.display='block'; ensureMap();
      if(ptsLayer){ try{ map.removeLayer(ptsLayer); } catch{} ptsLayer=null; }

      const polSel=selP.value;
      const gj=DATA[polSel];
      if(!gj || !gj.features || gj.features.length===0){ status.textContent=`Dados não encontrados para ${polSel}`; return; }

      // apenas valores significativos na legenda
      const signifVals=gj.features.map(f=>{
        const p=f.properties||{};
        const pv=p.p_value;
        if(pv===null||pv===undefined||pv===""||isNaN(Number(pv))) return null;
        if(Number(pv)>=0.05) return null;
        const v=('percent_change' in p)?Number(p.percent_change):NaN;
        return isNaN(v)?null:v;
      }).filter(v=>v!==null);

      if(signifVals.length===0){ status.textContent=`Nenhuma estação significativa (p<0.05) em ${polSel}`; return; }

      const minV=Math.min(...signifVals), maxV=Math.max(...signifVals);

      ptsLayer=L.geoJSON(gj,{
        pointToLayer:(f,latlng)=>{
          const p=f.properties||{};
          let pv=p.p_value;
          if(pv===null||pv===undefined||pv===""||isNaN(Number(pv))){
            return L.circleMarker(latlng,{radius:7,fillColor:"#ffffff",color:"#000000",weight:1,fillOpacity:0.95});
          }
          pv=Number(pv);
          const val=('percent_change' in p)?Number(p.percent_change):NaN;
          if(pv>=0.05) return L.circleMarker(latlng,{radius:7,fillColor:"#cccccc",color:"#222",weight:0.6,fillOpacity:0.95});
          const color=colorForPercent(val,minV,maxV), size=14, up=val>0;
          const html=up?`<svg width="${size}" height="${size}" viewBox="0 0 14 14"><polygon points="7,0 0,14 14,14" fill="${color}" stroke="black" stroke-width="1"/></svg>`:
                        `<svg width="${size}" height="${size}" viewBox="0 0 14 14"><polygon points="0,0 14,0 7,14" fill="${color}" stroke="black" stroke-width="1"/></svg>`;
          return L.marker(latlng,{icon:L.divIcon({className:'',iconSize:[size,size],html:html})});
        },
        onEachFeature:(f,layer)=>layer.bindPopup(buildPopup(f.properties||{}))
      }).addTo(map);

      layerControl.addOverlay(ptsLayer,"Estações (Mudança %)");
      addLegend(minV,maxV);

      const fg=L.featureGroup([ptsLayer]);
      if(fg.getBounds && fg.getBounds().isValid()) map.fitBounds(fg.getBounds(),{padding:[20,20]});
      status.textContent='';
    } catch(err){ console.error(err); status.textContent='Erro ao gerar o mapa (ver console)'; }
  });

})();
</script>
"""

html_code = html_template.replace("__DATA__", data_json)

HTML(html_code)

In [8]:
# Para salvar a figura e abrir no navegador, ajuste o caminho de saída conforme necessário.

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste output_path para o diretório/arquivo de saída no seu computador antes de executar o script.
Como o código é compartilhado via Git, este caminho varia entre usuários.
Certifique-se de que o diretório exista e que você tenha permissão de escrita. '''
output_dir = "outputs"
output_path = os.path.join(output_dir, "figura34.html")

# 1. Create the 'outputs' directory automatically if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# 2. Write the file
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

# 3. Open in browser
webbrowser.open(output_path)


True

### Tabela 21 - Tendências por UF e estação.
Tabela interativa e filtravel que resume os resultados das tendências dos poluentes CO, MP10, MP25, NO2, 03, SO2. O painel SearchPanes permite filtrar simultaneamente a coluna UF e ID_OEMA para visualização do resultado.

A tabela perimite realizar Download dos dados filtrados através da ferramenta interativa. 

 **Pré-requisitos:** Verificar a existência o arquivo `stations_trends.csv` na pasta `data` para gerar a `tabela_tendencias.html`

In [6]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import scripts.stationsTrendTableEx as table
import importlib
importlib.reload(table)

df, html = table.stations_trend_table_interactive(
    csv_name="/home/nobre/Notebooks/RQAR_2025_book/Guia_RQAr/data/outputs/stations_trends.csv",
    save_html=True,
    open_in_notebook=True
)

from IPython.display import IFrame
IFrame("../_static/tendencias/tabela_tendencias.html", width="100%", height=800)

FileNotFoundError: ❌ Arquivo não encontrado: /home/nobre/Notebooks/RQAR_2025_book/Guia_RQAr/data/outputs/stations_trends.csv